In [3]:
import numpy as np
import pygame


pygame 2.6.1 (SDL 2.28.4, Python 3.13.5)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [4]:
class Car:
    def __init__(self, x, y, direction):
        self.x = x
        self.y = y
        self.direction = direction  # "up", "down", "left", "right"
        self.speed = 2

    def move(self, can_move):
        if not can_move:
            return

        if self.direction == "up":
            self.y -= self.speed
        elif self.direction == "down":
            self.y += self.speed
        elif self.direction == "left":
            self.x -= self.speed
        elif self.direction == "right":
            self.x += self.speed

    def draw(self, screen):
        pygame.draw.rect(screen, (0, 0, 255), (self.x, self.y, 8, 8))


def can_move(direction, phase):
    # phase 0 = AC forward
    if phase == 0:
        return direction in ["up", "down"]

    # phase 1 = BD forward
    if phase == 1:
        return direction in ["left", "right"]

    # phase 2 = AC left (simplified same as forward here)
    if phase == 2:
        return direction in ["up", "down"]

    # phase 3 = BD left
    if phase == 3:
        return direction in ["left", "right"]

    return False

import random

def spawn_cars(cars):
    if random.random() < 0.2:
        cars.append(Car(400, 0, "down"))   # a
    if random.random() < 0.2:
        cars.append(Car(0, 400, "right"))  # b
    if random.random() < 0.2:
        cars.append(Car(400, 800, "up"))   # c
    if random.random() < 0.2:
        cars.append(Car(800, 400, "left")) # d

def count_cars(cars):
    counts = [0, 0, 0, 0]

    for car in cars:
        if car.direction == "down": counts[0] += 1
        elif car.direction == "right": counts[1] += 1
        elif car.direction == "up": counts[2] += 1
        elif car.direction == "left": counts[3] += 1

    return counts

import torch

def get_action(state, model):
    state_tensor = torch.FloatTensor(state).unsqueeze(0)  # shape (1, state_dim)

    with torch.no_grad():
        q_values = model(state_tensor)
        action = torch.argmax(q_values, dim=1).item()

    return action

In [7]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "code").exists():
    project_root = project_root.parent

if not (project_root / "code").exists():
    raise RuntimeError("Could not locate the project root containing the 'code' directory.")

source_root = str(project_root / "code")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

from State.traffic_env import TrafficEnv
from Neural_Networks.DQN_Implementation.dqn import DQN, ReplayBuffer



env = TrafficEnv()

state_dim = len(env._get_state())
action_dim = len(env.phases)


model = DQN(state_dim, action_dim)

model.load_state_dict(torch.load("../Neural_Networks/DQN_Implementation/traffic_dqn_model.pth", map_location=torch.device("cpu")))
model.eval()

DQN(
  (net): Sequential(
    (0): Linear(in_features=10, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=4, bias=True)
  )
)

In [ ]:
pygame.init()
screen = pygame.display.set_mode((800, 800))
clock = pygame.time.Clock()

cars = []
state = env.reset()

running = True
while running:
    screen.fill((255, 255, 255))

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            running = False

    # spawn cars
    spawn_cars(cars)

    # update state from simulation
    env.car_counts = np.array(count_cars(cars))

    # get action from model
    action = get_action(env._get_state(), model)
    env.current_phase = action

    # move cars
    for car in cars:
        move_allowed = can_move(car.direction, env.current_phase)
        car.move(move_allowed)

    # remove cars that exit screen
    cars = [c for c in cars if 0 <= c.x <= 800 and 0 <= c.y <= 800]

    # draw roads
    pygame.draw.rect(screen, (0,0,0), (350, 0, 100, 800))
    pygame.draw.rect(screen, (0,0,0), (0, 350, 800, 100))

    # draw cars
    for car in cars:
        car.draw(screen)

    # display phase
    font = pygame.font.SysFont(None, 30)
    text = font.render(f"Phase: {env.current_phase}", True, (255,0,0))
    screen.blit(text, (10, 10))

    pygame.display.flip()
    clock.tick(30)

: 